# 04 — Evaluation
Load trained classifiers and evaluate on the held-out test split (evaluated exactly once).

In [ ]:
import pickle, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from signalrag.config import TRAIN_CSV, TEST_CSV, FEATURE_COLS, LABEL_COL


In [ ]:
# Load classifiers first so we use their stored feat_cols
# (Issue 3 fix: feature-column alignment guaranteed by the trained classifier)
clfs = {}
for name in ["lr", "mlp", "rf", "xgb"]:
    with open(f"models/{name}.pkl", "rb") as f:
        clfs[name] = pickle.load(f)

# All four classifiers were trained on the same feat_cols; use the LR copy
feat_cols = clfs["lr"].feat_cols

test_df = pd.read_csv(TEST_CSV)

# Drop rows with any missing signal values (Issue 2 fix: mirror train-side dropna)
n_before = len(test_df)
test_df  = test_df[feat_cols + [LABEL_COL]].dropna().reset_index(drop=True)
if len(test_df) < n_before:
    print(f"  Dropped {n_before - len(test_df)} rows with missing values")

X_test = test_df[feat_cols].values
y_test = test_df[LABEL_COL].values
print(f"Test: {len(y_test)} | {int(y_test.sum())} faithful ({y_test.mean()*100:.1f}%)")


## Proposed models

In [ ]:
model_results = {}
for name in ["lr", "mlp", "rf", "xgb"]:
    clf = clfs[name]
    r = clf.evaluate(X_test, y_test)
    label = {"lr": "Logistic Regression", "mlp": "MLP",
             "rf": "Random Forest", "xgb": "XGBoost"}[name]
    model_results[f"Proposed: {label}"] = r
    print(f"  Proposed: {label:<22} AUROC={r['AUROC']:.3f}  F1={r['F1']:.3f}  Acc={r['Accuracy']:.3f}")


## Single-signal baselines (no supervision)

In [ ]:
baseline_specs = [
    ("Baseline: Entropy (inv.)",  "entropy",         True),
    ("Baseline: Avg Token Prob",  "avg_token_prob",   False),
    ("Baseline: Embed Sim",       "embed_sim",        False),
    ("Baseline: Retriever Score", "retriever_score",  False),
]
baseline_results = {}
X_train_raw = pd.read_csv(TRAIN_CSV)

for name, col, inv in baseline_specs:
    if col not in feat_cols:
        continue
    ci = feat_cols.index(col)
    tr = X_train_raw[col].values * (-1 if inv else 1)
    te = X_test[:, ci] * (-1 if inv else 1)
    mn, mx = tr.min(), tr.max()
    probs = np.clip((te - mn) / (mx - mn + 1e-9), 0, 1)
    preds = (probs >= 0.5).astype(int)
    baseline_results[name] = {
        "AUROC":    roc_auc_score(y_test, probs),
        "F1":       f1_score(y_test, preds, zero_division=0),
        "Accuracy": accuracy_score(y_test, preds),
    }
    print(f"  {name:<35} AUROC={baseline_results[name]['AUROC']:.3f}")


## Results table (Table 2)

In [ ]:
all_results = {**baseline_results, **model_results}
print(f"{'Model':<40} {'AUROC':>8} {'F1':>8} {'Acc':>8}")
print("-" * 70)
print(f"{'TPA [white-box ref]':<40} {'0.824':>8} {'0.601':>8} {'N/A':>8}")
print("-" * 70)
for name, r in all_results.items():
    print(f"{name:<40} {r['AUROC']:>8.3f} {r['F1']:>8.3f} {r['Accuracy']:>8.3f}")


## Feature importance (Table 4)

In [ ]:
with open("models/lr.pkl", "rb") as f:
    lr_clf = pickle.load(f)
print(f"{'Rank':<6} {'Signal':<30} {'|Coefficient|':>14}")
print("-" * 52)
for rank, (feat, imp) in enumerate(lr_clf.feature_importances(), 1):
    print(f"{rank:<6} {feat:<30} {imp:>14.3f}")
